In [5]:
# =============================================================================
# CELL 1 — INSTALL DEPENDENCIES
# =============================================================================
# !pip install duckdb groq plotly openpyxl -q


In [7]:
#CELL 2 — IMPORTS
# =============================================================================
import duckdb
import pandas as pd
import re
import uuid
import os
from groq import Groq
from IPython.display import display, Markdown, HTML
import plotly.express as px
import plotly.graph_objects as go

PLOTLY_AVAILABLE = True
print("Imports done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00
Imports done.


In [9]:
#CELL 3 — API KEY + CLIENT
# =============================================================================
import os
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready.")

Groq client ready.


In [10]:
# CELL 4 — LOAD DATA + DUCKDB SETUP
# =============================================================================
from google.colab import files

print("Please upload your Excel file...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print(f"Uploaded: {file_name}")

df_raw = pd.read_excel(file_name)

# Clean column names — remove spaces, brackets
df_raw.columns = [
    col.strip().replace(' ', '_').replace('(', '').replace(')', '')
    for col in df_raw.columns
]

# Save as Parquet
df_raw.to_parquet('nsdc_batches.parquet')

# DuckDB connection + view
duck_conn = duckdb.connect()
duck_conn.execute("CREATE VIEW nsdc_batches AS SELECT * FROM 'nsdc_batches.parquet'")

# Sanity check
test = duck_conn.execute("SELECT COUNT(*) as total_rows FROM nsdc_batches").df()
print(f"DuckDB ready. Rows: {test['total_rows'][0]}")
print(f"Columns: {list(df_raw.columns)}")


Please upload your Excel file...


Saving random data nsdc.xlsx to random data nsdc (1).xlsx
Uploaded: random data nsdc (1).xlsx
DuckDB ready. Rows: 10072
Columns: ['Batch_ID', 'TC_State', 'TC_District', 'TC_Constituency', 'TC_Smart_Id', 'TP_Smart_Id', 'Partner_Name', 'TC_Name', 'Job_Role_Code', 'Sector_Name', 'Job_Role', 'Job_Version', 'QPhours', 'Job_Level', 'Job_Role_Type', 'Common_Norms_Category', 'Submit_To_SSC_Date', 'Batch_Start_Date', 'Batch_End_Date', 'Batch_Start_Time', 'Batch_End_Time', 'Assessment_Date', 'OJT_Hours', 'Gender', 'Caste_Category', 'Religion', 'Minority', 'Differently_Abled', 'Enrolled', 'Dropout', 'Ongoing', 'Trained', 'Assessed', 'Passed', 'Failed', 'Not_Appeared', 'Certified', 'Reported_Placed', 'Self_Employed', 'Wage_Employed', 'Apprenticeship', 'Re_Assessed_Reassessment', 'Re_Passed_Reassessment', 'Re_Fail_Reassessment', 'Re_Not_Appeared_Reassessment', 'Re_Certified_Reassessment', 'Re_Placed_Reassessment', 'Re_Self_Employed_Reassessment', 'Re_Wage_Employed_Reassessment']


In [11]:
# CELL 5 — SCHEMA CONTEXT
# =============================================================================
SCHEMA_CONTEXT = """
You are an expert data analyst for NSDC (National Skill Development Corporation).
You have access to a single table called `nsdc_batches` with ~10,000 rows.
Each row represents a training batch record broken down by gender and category.

TABLE: nsdc_batches

--- IDENTIFIERS (internal only, never expose in results) ---
Batch_ID          TEXT    -- unique batch identifier
TC_Smart_Id       REAL    -- training centre system ID
TP_Smart_Id       INT     -- training partner system ID
Job_Role_Code     TEXT    -- coded job role identifier

--- LOCATION ---
TC_State          TEXT    -- state where training centre is located
TC_District       TEXT    -- district (some nulls)
TC_Constituency   TEXT    -- constituency (some nulls)

--- PARTNER/CENTRE INFO (RESTRICTED — never expose in results) ---
Partner_Name      TEXT    -- name of training partner organisation
TC_Name           TEXT    -- name of training centre

--- TRAINING DETAILS ---
Sector_Name            TEXT  -- industry sector (e.g. MEDIA & ENTERTAINMENT)
Job_Role               TEXT  -- specific job role being trained for
Job_Version            REAL  -- version of job role curriculum
Job_Level              REAL  -- level of the job role
Job_Role_Type          TEXT  -- type of job role
Common_Norms_Category  TEXT  -- norms category (I, II, III etc.)
QPhours                INT   -- qualification pack hours
OJT_Hours              REAL  -- on-job training hours (many nulls, use COALESCE(OJT_Hours,0))

--- DATES ---
Submit_To_SSC_Date  DATE  -- date batch submitted to SSC
Batch_Start_Date    DATE  -- training batch start date
Batch_End_Date      DATE  -- training batch end date
Assessment_Date     DATE  -- date of assessment
Batch_Start_Time    TEXT  -- start time
Batch_End_Time      TEXT  -- end time

--- DEMOGRAPHICS ---
Gender            TEXT  -- Male or Female
Caste_Category    TEXT  -- Gen, OBC, SC, ST etc. (some nulls)
Religion          TEXT  -- religion (many nulls)
Minority          TEXT  -- Yes or No
Differently_Abled TEXT  -- Yes or No

--- OUTCOME COUNTS (use SUM for aggregations, not AVG — each row is already a count) ---
Enrolled          INT  -- candidates enrolled
Dropout           INT  -- dropped out
Ongoing           INT  -- still in training
Trained           INT  -- completed training
Assessed          INT  -- appeared for assessment
Passed            INT  -- passed assessment
Failed            INT  -- failed assessment
Not_Appeared      INT  -- did not appear for assessment
Certified         INT  -- received certification
Reported_Placed   INT  -- placed in jobs
Self_Employed     INT  -- became self employed
Wage_Employed     INT  -- in wage employment
Apprenticeship    INT  -- in apprenticeship

--- REASSESSMENT OUTCOMES ---
Re_Assessed_Reassessment      INT
Re_Passed_Reassessment        INT
Re_Fail_Reassessment          INT
Re_Not_Appeared_Reassessment  INT
Re_Certified_Reassessment     INT
Re_Placed_Reassessment        INT
Re_Self_Employed_Reassessment INT
Re_Wage_Employed_Reassessment INT

--- QUERY RULES ---
1. Always use exact column names with underscores as listed above.
2. For aggregations use SUM() on outcome counts, not AVG().
3. Use strftime('%Y', Batch_Start_Date) to extract year from dates.
4. Use COALESCE(OJT_Hours, 0) when referencing OJT_Hours.
5. Never expose Partner_Name, TC_Name, TC_Smart_Id, TP_Smart_Id, Batch_ID, Job_Role_Code.
6. Use LIKE for partial string matches on state/sector names.
7. Always write a valid DuckDB SELECT query. No markdown, no explanation.
"""

print("SCHEMA_CONTEXT set.")

SCHEMA_CONTEXT set.


In [31]:
#CELL 6 — GUARDRAILS
# =============================================================================
QUESTION_KEYWORDS = [
    'state', 'district', 'constituency', 'region', 'zone',
    'sector', 'job', 'role', 'batch', 'training', 'centre', 'center',
    'partner', 'course', 'skill', 'level', 'hours', 'qp',
    'enrolled', 'dropout', 'trained', 'assessed', 'passed', 'failed',
    'certified', 'placed', 'employed', 'apprenticeship', 'ongoing',
    'appeared', 'reassess',
    'gender', 'male', 'female', 'caste', 'religion', 'minority', 'abled',
    'sc', 'st', 'obc', 'general',
    'date', 'year', 'month', '2023', '2024', '2022',
    'how many', 'total', 'count', 'average', 'which', 'top', 'most',
    'least', 'compare', 'breakdown', 'distribution', 'trend', 'rate',
    'nsdc', 'candidate', 'batch', 'sector', 'pass', 'fail'
]

PII_TERMS = [
    'partner_name', 'partner name', 'tc_name', 'tc name',
    'training centre name', 'training center name',
    'tc_smart_id', 'tp_smart_id', 'smart id',
    'batch_id', 'batch id', 'job_role_code',
    'phone', 'mobile', 'email', 'address',
    'contact', 'who is', 'identity', 'personal'
]

OFFENSIVE_TERMS = [
    'stupid', 'idiot', 'dumb', 'hate', 'kill', 'abuse',
    'racist', 'sexist', 'porn', 'sex', 'nude', 'violence',
    'terrorist', 'illegal', 'fraud', 'corrupt'
]

def is_database_question(question: str) -> bool:
    q = question.lower()
    return any(keyword in q for keyword in QUESTION_KEYWORDS)

def is_pii_request(question: str) -> bool:
    q = question.lower()
    return any(term in q for term in PII_TERMS)

def is_offensive(question: str) -> bool:
    q = question.lower()
    return any(term in q for term in OFFENSIVE_TERMS)

print("Guardrails set.")

Guardrails set.


In [17]:
# CELL 7 — CORE SQL FUNCTIONS
# =============================================================================
def clean_sql(raw: str) -> str:
    sql = raw.strip()
    # Remove markdown fences
    sql = re.sub(r"^```(?:sql)?", "", sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"```$", "", sql).strip()
    # Cut at first semicolon (kills stacked statements)
    sql = sql.split(";", 1)[0].strip()
    return sql

def validate_sql(sql: str) -> None:
    lowered = sql.lower().strip()
    blocked = ['insert', 'update', 'delete', 'drop', 'alter',
               'create', 'replace', 'truncate', 'pragma']
    if not lowered.startswith("select"):
        raise ValueError("Only SELECT queries are allowed.")
    if any(re.search(rf"\b{word}\b", lowered) for word in blocked):
        raise ValueError("Query contains a blocked keyword.")

def generate_sql(question: str) -> str:
    prompt = f"""
{SCHEMA_CONTEXT}

User question: {question}

Return a valid DuckDB SELECT query only. No explanation, no markdown, no backticks.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    sql = clean_sql(response.choices[0].message.content)
    validate_sql(sql)
    return sql

def repair_sql(question: str, bad_sql: str, error_message: str) -> str:
    prompt = f"""
You are a careful DuckDB SQL expert.

{SCHEMA_CONTEXT}

The SQL below failed. Fix it.

User question: {question}
Failed SQL: {bad_sql}
Error: {error_message}

Return a corrected DuckDB SELECT query only. No explanation, no markdown.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    fixed_sql = clean_sql(response.choices[0].message.content)
    validate_sql(fixed_sql)
    return fixed_sql

def execute_sql_duckdb(sql: str) -> pd.DataFrame:
    return duck_conn.execute(sql).df()

print("SQL functions ready.")

SQL functions ready.


In [19]:
 #CELL 8 — BI OUTPUT FUNCTIONS
# =============================================================================
def generate_kpis(df: pd.DataFrame) -> list:
    kpis = []
    kpis.append({"label": "Rows returned", "value": len(df)})
    kpis.append({"label": "Columns returned", "value": len(df.columns)})
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    for col in numeric_cols[:4]:
        series = df[col].dropna()
        if len(series) > 0:
            kpis.append({"label": f"Total {col}", "value": int(series.sum())})
            kpis.append({"label": f"Average {col}", "value": round(float(series.mean()), 2)})
    return kpis

def display_kpis(kpis: list) -> None:
    cards_html = "<div style='display:flex;flex-wrap:wrap;gap:12px;margin:12px 0;'>"
    for kpi in kpis:
        cards_html += f"""
        <div style='background:#1a1a2e;border:1px solid #4a90d9;border-radius:8px;
                    padding:16px 20px;min-width:160px;text-align:center;'>
            <div style='color:#4a90d9;font-size:11px;font-weight:600;
                        text-transform:uppercase;letter-spacing:1px;margin-bottom:6px;'>
                {kpi['label']}
            </div>
            <div style='color:#ffffff;font-size:22px;font-weight:700;'>
                {kpi['value']:,} if isinstance(kpi['value'], int) else {kpi['value']}
            </div>
        </div>"""
    cards_html += "</div>"
    display(HTML(cards_html))

def choose_chart_specs(df: pd.DataFrame) -> list:
    if df.empty:
        return []
    specs = []
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    low_cardinality_numeric = [
        col for col in numeric_cols
        if df[col].nunique(dropna=True) <= 20 and len(df) > 1
    ]
    text_cols = [col for col in df.columns if col not in numeric_cols]
    categorical_cols = text_cols + low_cardinality_numeric

    for cat in categorical_cols:
        metric_candidates = [col for col in numeric_cols if col != cat]
        if metric_candidates:
            for metric in metric_candidates[:2]:
                specs.append({
                    "type": "bar", "x": cat, "y": metric,
                    "title": f"{metric} by {cat}"
                })
            return specs[:2]

    if text_cols:
        cat = text_cols[0]
        counts = df[cat].value_counts(dropna=False).reset_index()
        counts.columns = [cat, "count"]
        if len(counts) <= 8:
            specs.append({"type": "pie", "source_df": counts,
                         "names": cat, "values": "count",
                         "title": f"Distribution of {cat}"})
        else:
            specs.append({"type": "bar", "source_df": counts.head(20),
                         "x": cat, "y": "count",
                         "title": f"Top {cat} values"})

    if not specs and numeric_cols and len(df) > 1:
        specs.append({"type": "histogram", "x": numeric_cols[0],
                     "title": f"Distribution of {numeric_cols[0]}"})
    return specs[:2]

def render_charts(df: pd.DataFrame, specs: list) -> None:
    if not specs:
        display(Markdown("_No chart generated — result too small or not chart-friendly._"))
        return
    for spec in specs:
        source = spec.get("source_df", df)
        try:
            if spec["type"] == "bar":
                fig = px.bar(source, x=spec["x"], y=spec["y"],
                           title=spec["title"],
                           template="plotly_dark",
                           color=spec["y"],
                           color_continuous_scale="Blues")
            elif spec["type"] == "pie":
                fig = px.pie(source, names=spec["names"], values=spec["values"],
                           title=spec["title"], template="plotly_dark",
                           hole=0.35)
            elif spec["type"] == "histogram":
                fig = px.histogram(source, x=spec["x"],
                                 title=spec["title"],
                                 template="plotly_dark")
            else:
                continue

            fig.update_layout(
                plot_bgcolor='rgba(0,0,0,0)',
                paper_bgcolor='rgba(26,26,46,1)',
                font_color='white',
                title_font_size=16,
                margin=dict(t=50, l=40, r=40, b=40)
            )
            fig.show()
        except Exception as e:
            display(Markdown(f"_Chart error: {e}_"))

def generate_human_answer(question: str, sql: str, df: pd.DataFrame) -> str:
    prompt = f"""
The user asked: {question}
SQL used: {sql}
Result preview: {df.head(5).to_string()}

Write one clear, plain English sentence summarising the key finding.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

def generate_insights(question: str, sql: str, df: pd.DataFrame) -> str:
    prompt = f"""
The user asked: {question}
SQL used: {sql}
Result: {df.head(10).to_string()}

Write 2-3 short, business-style insights about this data.
Do NOT invent numbers not present in the result above.
Be specific and factual.
"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

def export_results(df: pd.DataFrame) -> dict:
    uid = uuid.uuid4().hex[:8]
    csv_path = f"query_result_{uid}.csv"
    excel_path = f"query_result_{uid}.xlsx"
    df.to_csv(csv_path, index=False)
    df.to_excel(excel_path, index=False)
    return {"csv": csv_path, "excel": excel_path}

def show_download_links(downloads: dict) -> None:
    display(Markdown(f"📥 Results saved: `{downloads['csv']}` and `{downloads['excel']}`"))

print("BI output functions ready.")

BI output functions ready.


In [21]:
# CELL 9 — MAIN PIPELINE FUNCTION
# =============================================================================
def ask_nsdc(question: str, show_table: bool = True, debug: bool = False) -> dict:

    # Guardrail 1: offensive language
    if is_offensive(question):
        display(Markdown("⚠️ Your question contains inappropriate language. Please rephrase."))
        return {"question": question, "error": "Offensive language detected."}

    # Guardrail 2: PII request
    if is_pii_request(question):
        display(Markdown("🔒 This question requests restricted information that cannot be shared."))
        return {"question": question, "error": "PII request blocked."}

    # Guardrail 3: relevance
    if not is_database_question(question):
        display(Markdown("ℹ️ I can only answer questions related to the NSDC training dataset."))
        return {"question": question, "error": "Not a database-related question."}

    try:
        sql = generate_sql(question)
        repaired = False

        try:
            result_df = execute_sql_duckdb(sql)
        except Exception as db_error:
            original_sql = sql
            sql = repair_sql(question, original_sql, str(db_error))
            result_df = execute_sql_duckdb(sql)
            repaired = True

        downloads = export_results(result_df)
        kpis = generate_kpis(result_df)
        chart_specs = choose_chart_specs(result_df)
        answer = generate_human_answer(question, sql, result_df)
        insights = generate_insights(question, sql, result_df)

        display(Markdown("## 🤖 NSDC Analytics Result"))
        display(Markdown(f"**Question:** {question}"))
        display(Markdown("**Generated SQL:**"))
        display(Markdown(f"```sql\n{sql}\n```"))
        if repaired:
            display(Markdown("_⚠️ SQL was auto-repaired after an error._"))
        display(Markdown(f"**Answer:** {answer}"))
        display(Markdown("### 📊 KPI Summary"))
        display_kpis(kpis)
        display(Markdown("### 📋 Query Result"))
        if show_table:
            display(result_df)
        show_download_links(downloads)
        display(Markdown("### 📈 Visualizations"))
        render_charts(result_df, chart_specs)
        display(Markdown("### 💡 AI Insights"))
        display(Markdown(insights))

        if debug:
            print("SQL:", sql)
            print("Shape:", result_df.shape)
            print("Repaired:", repaired)

        return {
            "question": question, "sql": sql,
            "dataframe": result_df, "answer": answer,
            "kpis": kpis, "chart_specs": chart_specs,
            "insights": insights, "downloads": downloads,
            "sql_repaired": repaired
        }

    except Exception as e:
        display(Markdown(f"⚠️ Query could not be completed: `{str(e)}`"))
        return {"question": question, "error": str(e)}

print("Pipeline ready. Use ask_nsdc('your question here')")

Pipeline ready. Use ask_nsdc('your question here')


In [25]:
ask_nsdc('You stupid system show me data')

⚠️ Your question contains inappropriate language. Please rephrase.

{'question': 'You stupid system show me data',
 'error': 'Offensive language detected.'}

In [26]:
ask_nsdc("What is the partner name for batch ir16767?")

🔒 This question requests restricted information that cannot be shared.

{'question': 'What is the partner name for batch ir16767?',
 'error': 'PII request blocked.'}

In [27]:
ask_nsdc("What is the capital of France?")

ℹ️ I can only answer questions related to the NSDC training dataset.

{'question': 'What is the capital of France?',
 'error': 'Not a database-related question.'}

In [28]:
ask_nsdc("How many candidates were enrolled and passed in each sector?")

## 🤖 NSDC Analytics Result

**Question:** How many candidates were enrolled and passed in each sector?

**Generated SQL:**

```sql
SELECT Sector_Name, SUM(Enrolled) AS Total_Enrolled, SUM(Passed) AS Total_Passed FROM nsdc_batches GROUP BY Sector_Name
```

**Answer:** The Electronics sector had the highest number of enrolled and passed candidates, with 7,167 enrolled and 2,098 passing, among the different sectors.

...

📥 Results saved: `query_result_04f00e88.csv` and `query_result_04f00e88.xlsx`

### 📈 Visualizations

### 💡 AI Insights

Based on the data, here are 3 short business-style insights:

1. The Electronics sector has the highest number of enrolled candidates, with a total of 7,167, and also has one of the highest pass rates, with 2,098 candidates passing.
2. The Media & Entertainment sector has the second-highest number of enrolled candidates, with 3,491, and also has the second-highest number of passed candidates, with 1,846.
3. The Rubber sector has the lowest pass rate, with only 37 candidates passing out of 1,200 enrolled, indicating a potential area for improvement in this sector.

{'question': 'How many candidates were enrolled and passed in each sector?',
 'sql': 'SELECT Sector_Name, SUM(Enrolled) AS Total_Enrolled, SUM(Passed) AS Total_Passed FROM nsdc_batches GROUP BY Sector_Name',
 'dataframe':               Sector_Name  Total_Enrolled  Total_Passed
 0             ELECTRONICS          7167.0        2098.0
 1                 TELECOM          4036.0        1139.0
 2              GREEN JOBS          1949.0         672.0
 3                 TEXTILE          1210.0         316.0
 4                  RUBBER          1200.0          37.0
 5              MANAGEMENT           725.0         153.0
 6            CONSTRUCTION           782.0         139.0
 7                PLUMBING           120.0          31.0
 8   TOURISM & HOSPITALITY          1040.0         332.0
 9   MEDIA & ENTERTAINMENT          3491.0        1846.0
 10        FOOD PROCESSING          2268.0         450.0
 11                LEATHER           406.0         178.0
 12      BEAUTY & WELLNESS          37

In [32]:
ask_nsdc("Which state has the highest number of certified candidates?")

## 🤖 NSDC Analytics Result

**Question:** Which state has the highest number of certified candidates?

**Generated SQL:**

```sql
SELECT TC_State, SUM(Certified) FROM nsdc_batches GROUP BY TC_State ORDER BY SUM(Certified) DESC LIMIT 1
```

**Answer:** Uttar Pradesh has the highest number of certified candidates, with a total of 7835.

...

📥 Results saved: `query_result_33c8a2ff.csv` and `query_result_33c8a2ff.xlsx`

### 📈 Visualizations

### 💡 AI Insights

Based on the data, here are 2-3 short business-style insights:

1. Uttar Pradesh has the highest number of certified candidates, indicating a strong presence of training centers or a high demand for certification programs in the state.
2. The total number of certified candidates in Uttar Pradesh is 7835, suggesting a significant investment in workforce development and skills training in the region.
3. The state of Uttar Pradesh accounts for the largest share of certified candidates, implying that it may be a key market for organizations offering certification programs or related services.

{'question': 'Which state has the highest number of certified candidates?',
 'sql': 'SELECT TC_State, SUM(Certified) FROM nsdc_batches GROUP BY TC_State ORDER BY SUM(Certified) DESC LIMIT 1',
 'dataframe':         TC_State  sum(Certified)
 0  Uttar Pradesh          7835.0,
 'answer': 'Uttar Pradesh has the highest number of certified candidates, with a total of 7835.',
 'kpis': [{'label': 'Rows returned', 'value': 1},
  {'label': 'Columns returned', 'value': 2},
  {'label': 'Total sum(Certified)', 'value': 7835},
  {'label': 'Average sum(Certified)', 'value': 7835.0}],
 'chart_specs': [{'type': 'bar',
   'x': 'TC_State',
   'y': 'sum(Certified)',
   'title': 'sum(Certified) by TC_State'}],
 'insights': 'Based on the data, here are 2-3 short business-style insights:\n\n1. Uttar Pradesh has the highest number of certified candidates, indicating a strong presence of training centers or a high demand for certification programs in the state.\n2. The total number of certified candidates in U

In [30]:
ask_nsdc("Show me the breakdown of enrolled candidates by gender")

## 🤖 NSDC Analytics Result

**Question:** Show me the breakdown of enrolled candidates by gender

**Generated SQL:**

```sql
SELECT Gender, SUM(Enrolled) AS Total_Enrolled FROM nsdc_batches GROUP BY Gender
```

**Answer:** There are slightly more female candidates enrolled, with 22,855 females compared to 22,626 males.

...

📥 Results saved: `query_result_4dda30b3.csv` and `query_result_4dda30b3.xlsx`

### 📈 Visualizations

### 💡 AI Insights

Based on the enrollment data, here are 3 key insights:

1. The total number of female candidates enrolled is 22855, which is slightly higher than their male counterparts.
2. Male candidates account for 22626 enrollments, indicating a near-equitable distribution of enrollments between genders.
3. The enrollment numbers suggest that the program has achieved a relatively balanced gender representation, with females making up a marginally larger proportion of total enrollments.

{'question': 'Show me the breakdown of enrolled candidates by gender',
 'sql': 'SELECT Gender, SUM(Enrolled) AS Total_Enrolled FROM nsdc_batches GROUP BY Gender',
 'dataframe':    Gender  Total_Enrolled
 0  Female         22855.0
 1    Male         22626.0,
 'answer': 'There are slightly more female candidates enrolled, with 22,855 females compared to 22,626 males.',
 'kpis': [{'label': 'Rows returned', 'value': 2},
  {'label': 'Columns returned', 'value': 2},
  {'label': 'Total Total_Enrolled', 'value': 45481},
  {'label': 'Average Total_Enrolled', 'value': 22740.5}],
 'chart_specs': [{'type': 'bar',
   'x': 'Gender',
   'y': 'Total_Enrolled',
   'title': 'Total_Enrolled by Gender'}],
 'insights': 'Based on the enrollment data, here are 3 key insights:\n\n1. The total number of female candidates enrolled is 22855, which is slightly higher than their male counterparts.\n2. Male candidates account for 22626 enrollments, indicating a near-equitable distribution of enrollments between gen